# Initial Info

* NOTE CALIBRATION INFO: (always adapt and save as new file, if new calibration is performed)
* Name: David Tiede
* Calibration source: Krypton lamp, persistent lines from 770 to 880nm.
* Spectral lines taken from: https://pml.nist.gov/PhysRefData/Handbook/Tables/kryptontable2_a.htm
* Calibration date: 2025_09_26

* This calibration procedure was copied and adapted from https://github.com/ScopeFoundry/HW_acton_spec/blob/master/spectrometer_calibration.ipynb

# Calibration Procedure
* Follow the steps in this Jupyther notebook to calibrate any Princeton Instruments spectrograph.


1. Compute center offset:
 - Set $\lambda_{\rm center}$ to set of known spectral lines
 - Measure pixel position of each spectral line. The position should be roughly centered on the CCD. 
 - average each to determine central pixel $n_o$
 
|   $\lambda_{\rm center}$ | Pixel |           
| ----------------------:  |:------:|
| 0   nm                   | 508 | 
| 445 nm                   | 507     |  
| 901 nm                   | 509      | 


2. Compute spectrometer calibration angles/length ($\ f_L, \delta, \gamma$)
 * Move known spectral line $\lambda_o$ to left and right sides as well as to the center of the detector
 * record $\lambda_{\rm center}$ and pixel position for each 
 * Compute best fit of $\ f_{\rm calib}$


| $\lambda_o$   | Side | $\lambda_{\rm center}$| Pixel  |
| ------------- | ---- |:----------------------|-------:|
| 365.02 nm      | R    |265.02 nm            |905     |
| 365.02 nm      | C    |365.02 nm            |508     |
| 365.02 nm      | L    |465.02 nm            |103     |
| 404.66 nm      | R    |304.66 nm            |909     |
| ...           | ...  | ...                   |...     |

3. Check if calibration is satisfactory by plotting the residual of the fit values. Adapt initial values and data if needed.

4. SAVE notebook with corresponding name to be transparent with your colleagues! 

# Optimization Function

* This section gives a short theoretical overview of the optimization function. Just acknowledge it and continue below. 
Optimize for 3 parameters:
 * $f_L$: Focal length of spectrometer
 * $\delta$: Detector angle (The angle of the image plane relative to the plane perpendicular to the spectrograph focal axis at the center of the image plane)
 * $\gamma$: inclusion angle

from experiment:
 * $n =  n_{px} - n_o$: Pixel from central pixel
 * $\lambda_{\rm center}$: Wavelength of center pixel 
 * $\lambda_p$: Wavelength of pixel $n$
 
Fixed Constants:
 * $m$: Diffraction order (typically one)
 * $x_{\rm pixel}$: pixel size
 * $d_{grating}$: Grating pitch (1/(groves / mm))
    
residual: (wl,  wl_p, n, f, delta,gamma)

We measure pixel position ($n$) of a known wavelength ($\lambda_p$) for multple peaks and spectrometer positions and find the best fit parameters $\ f_L, \delta, \gamma$:

$$ \lambda_p = f_{\rm calib} ( n,  \lambda_{\rm center}, 
    \underbrace{m, x_{\rm pixel}, d_{\rm grating}}_{\rm spec\ params}, 
    \overbrace{f_L,\ \ \delta,\ \ \gamma}^{\rm Calibration\ params} ) $$

$$ \lambda_p = \frac{d}{m} \cdot \left[ \sin( \psi - \frac{\gamma}{2}) + \sin(\psi+\frac{\gamma}{2} + \eta) \right]$$

Where

$$ \psi = \arcsin \left[ \frac{ m\ \lambda_{\rm center} } { 2\ d_{\rm grating} \cos(\frac{\gamma}{2})} \right] $$

$$ \eta = \arctan \left[ \frac{ n\ x_{pixel} \cos{\delta}} {f_L + n\ x_{pixel} \sin(\delta)} \right]$$

$$n =  n_{px} - n_o$$



In [ ]:
from __future__ import division
import numpy as np
import  matplotlib.pyplot as plt
from pprint import pprint
import numpy as np

# matplotlib notebook

In [ ]:
def wl_p_calib_p2(px, n0, offset_adjust, wl_center, m_order, d_grating, x_pixel, f, delta, gamma, curvature):
    #consts
    #d_grating = 1./150. #mm
    #x_pixel   = 16e-3 # mm
    #m_order   = 1 # diffraction order, unitless
    n = px - (n0+offset_adjust*wl_center)

    psi = np.arcsin( m_order* wl_center / (2*d_grating*np.cos(gamma/2.)))
    eta = np.arctan(n*x_pixel*np.cos(delta) / (f+n*x_pixel*np.sin(delta)))
    
    return ((d_grating/m_order)
                    *(np.sin(psi-0.5*gamma)
                      + np.sin(psi+0.5*gamma+eta))) + curvature*n**2

In [ ]:
from scipy.optimize import least_squares

def fit_residual(
                # optimization parameters
                opt_params, #  (f, delta, gamma, curvature)
                # other params and data
                px, n0, offset_adjust, wl_center, m_order, d_grating, x_pixel,
                wl_actual
                ):
    
    (f, delta, gamma, curvature) = opt_params
    wl_model = wl_p_calib_p2(px, n0, offset_adjust, wl_center, m_order, d_grating, x_pixel, f, delta, gamma,curvature)
    return wl_model - wl_actual

# grating 3 (600 g/mm Bz 500)

In [ ]:
# grating 3 (600 g/mm Bz 500)

# NOTE CALIBRATION INFO:
# Name: David Tiede
# Calibration source: Krypton lamp, persistent lines of Hg I from 770 to 880nm.
# Spectral lines taken from: https://pml.nist.gov/PhysRefData/Handbook/Tables/kryptontable2_a.htm
# Calibration date: 2025_09_26
wl_center_data = np.array([
[760.2, 243],
[811.3, 241],
[829.8, 244],
[877.7, 244],
])
    
n0 = np.mean(wl_center_data[:,1])
n0    
plt.figure()
plt.plot(wl_center_data[:,0], wl_center_data[:,1])
plt.axhline(n0)

In [ ]:
dispersion_data = np.array([
#wl_actual, wl_center, pixel
[760.2,760.2, 243],
[760.2,720 ,470],
[760.2,810 ,17],
[811.3,811.3,241],
[811.3, 771,472],
[811.3, 851,16],
[829.8,829.8 ,244],
[829.8, 790,470],
[829.8, 870,13],
[877.7,877.7,244],
[877.7,837,477],
[877.7,917,17],
])


In [ ]:
# MAKE SURE THE INITIAL PARAMETERS ARE GOOD !!! CHECK RESIDUAL FOR THE GOOD FITTING and adapt manually if required.
initial_guess = (150e6,0.9,0.5,0)

kwargs = dict(
    px=dispersion_data[:,2], 
    n0=np.mean(wl_center_data[:,1]),
    wl_center=dispersion_data[:,1], # nm
    m_order=1,
    d_grating=1/150.*1e6, # nm
    x_pixel=26e3, #nm
    wl_actual=dispersion_data[:,0], # nm
    offset_adjust = 0
)
result = least_squares(fit_residual, initial_guess, kwargs=kwargs)
result.x

In [ ]:
kwargs = dict(
    px=dispersion_data[:,2], 
    #px=wl_center_data[:,1],
    n0=np.mean(wl_center_data[:,1]),
    wl_center=dispersion_data[:,1],
    #wl_center=wl_center_data[:,0]*1e-6,
    m_order=1,
    d_grating=1/150.*1e6, # nm
    x_pixel=26e3, #nm
    #wl_actual=dispersion_data[:,0]*1e-6,
    f = result.x[0],
    delta = result.x[1],
    gamma = result.x[2],
    offset_adjust = 0,
    curvature = result.x[3]
)

wl_p_calib_p2(**kwargs) - dispersion_data[:,0]

In [ ]:
# to store for ini file
# f, delta, gamma, n0, offset_adjust, d_grating, x_pixel, curvature
Y = 'f, delta, gamma, n0, offset_adjust, d_grating, x_pixel, curvature'.split(', ')
str([ kwargs[x] for x in Y ])
#kwargs

In [ ]:
fig = plt.figure()
f, delta, gamma, n0, offset_adjust, d_grating, x_pixel, curvature = [np.float64(330605663.74965495), np.float64(-0.20488367116307532), np.float64(2.021864300924973), np.float64(508.0), 0, 6666.666666666667, 26000.0, np.float64(3.1224154313329654e-06)]
residual = wl_p_calib_p2(dispersion_data[:,2], n0, offset_adjust, dispersion_data[:,1], 1, d_grating, x_pixel, f, delta, gamma, curvature)-dispersion_data[:,0]

#plt.plot(wl_p_calib_p2(**kwargs))
plt.plot(residual,'.')
plt.title("Fit residuals")
plt.show()